# Local Gemini School-Region GeoJSON Generator

This notebook only runs the Gemini LLM call and exports a QGIS-ready GeoJSON from `sample_school_contexts.geojson`.

Outputs are written to `outputs/local_guy_ai/` with:
- School point feature
- School-related road `LineString` features
- School region polygon with border-only style (`fill_opacity = 0`)

In [1]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

ROOT = Path.cwd().resolve()
if not (ROOT / "sample_school_contexts.geojson").exists() and (ROOT.parent / "sample_school_contexts.geojson").exists():
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env", override=False)

INPUT_GEOJSON = ROOT / "sample_school_contexts.geojson"
OUTPUT_DIR = ROOT / "outputs" / "local_guy_ai"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_GEOJSON = OUTPUT_DIR / "school_related_roads.geojson"
OUTPUT_RESPONSE = OUTPUT_DIR / "gemini_school_region_response.json"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")

print(f"Input: {INPUT_GEOJSON}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Model: {GEMINI_MODEL}")

k:\Tech\Hackathon\Here-Berlin-2026\Sign_Detection\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Input: K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detection\sample_school_contexts.geojson
Output folder: K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detection\outputs\local_guy_ai
Model: gemini-3-flash-preview


In [2]:
def get_gemini_api_key() -> str:
    for key_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        value = os.getenv(key_name)
        if value:
            return value
    raise EnvironmentError("Set GOOGLE_API_KEY or GEMINI_API_KEY in .env")


def read_geojson(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def summarize_feature(feature: Dict[str, Any], index: int) -> Dict[str, Any]:
    geometry = feature.get("geometry", {})
    properties = feature.get("properties", {})
    summary = {
        "index": index,
        "feature_type": properties.get("feature_type"),
        "source": properties.get("source"),
        "road_name": properties.get("road_name"),
        "travel_direction": properties.get("travel_direction"),
        "average_speed": properties.get("average_speed"),
        "schoolzone_gfrgroup": properties.get("schoolzone_gfrgroup"),
        "speedlimit_gfrgroup": properties.get("speedlimit_gfrgroup"),
        "geometry_type": geometry.get("type"),
    }
    coords = geometry.get("coordinates", [])
    if geometry.get("type") == "Point":
        summary["coordinates"] = coords
    elif geometry.get("type") == "LineString":
        summary["coordinates"] = {
            "start": coords[0] if coords else None,
            "end": coords[-1] if coords else None,
            "point_count": len(coords),
        }
    return summary


def build_prompt(input_geojson: Dict[str, Any]) -> str:
    catalog = [summarize_feature(feature, i) for i, feature in enumerate(input_geojson.get("features", []))]
    return f"""You are a precise geospatial extraction assistant.

Task: identify the school region and all school-related roads from the provided feature catalog.

Requirements:
1. Return only strict JSON.
2. Identify one school point index.
3. Select all school-related road LineString indices.
4. Include optional supporting feature indices (such as speed signs).
5. Return a polygon for the school region if possible.

Schema to return exactly:
{{
  \"school_label\": string,
  \"school_feature_index\": integer | null,
  \"selected_road_feature_indices\": [integer, ...],
  \"selected_supporting_feature_indices\": [integer, ...],
  \"school_region_polygon\": {{
    \"type\": \"Polygon\",
    \"coordinates\": [[[lon, lat], ...]]
  }} | null,
  \"confidence\": number,
  \"reasoning\": string,
  \"notes\": [string, ...]
}}

Feature catalog:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
"""


def normalize_llm_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: List[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def extract_json_object(text: str) -> Dict[str, Any]:
    candidate = text.strip()
    if candidate.startswith("```"):
        candidate = re.sub(r"^```(?:json)?\s*", "", candidate, flags=re.IGNORECASE)
        candidate = re.sub(r"\s*```$", "", candidate)
    start = candidate.find("{")
    end = candidate.rfind("}")
    if start < 0 or end < 0 or end <= start:
        raise ValueError("Model did not return a JSON object")
    return json.loads(candidate[start : end + 1])


def fallback_polygon(input_geojson: Dict[str, Any], road_indices: List[int], school_index: Optional[int]) -> Optional[Dict[str, Any]]:
    features = input_geojson.get("features", [])
    points: List[Tuple[float, float]] = []
    chosen = list(dict.fromkeys([*road_indices, school_index] if school_index is not None else road_indices))
    for idx in chosen:
        if idx is None or idx < 0 or idx >= len(features):
            continue
        geom = features[idx].get("geometry", {})
        gtype = geom.get("type")
        coords = geom.get("coordinates", [])
        if gtype == "Point" and len(coords) >= 2:
            points.append((float(coords[0]), float(coords[1])))
        elif gtype == "LineString":
            for lon, lat in coords:
                points.append((float(lon), float(lat)))
    if not points:
        return None
    min_lon = min(lon for lon, _ in points)
    max_lon = max(lon for lon, _ in points)
    min_lat = min(lat for _, lat in points)
    max_lat = max(lat for _, lat in points)
    pad = 0.00045
    return {
        "type": "Polygon",
        "coordinates": [[
            [min_lon - pad, min_lat - pad],
            [max_lon + pad, min_lat - pad],
            [max_lon + pad, max_lat + pad],
            [min_lon - pad, max_lat + pad],
            [min_lon - pad, min_lat - pad],
        ]],
    }


def build_output_geojson(input_geojson: Dict[str, Any], llm_result: Dict[str, Any]) -> Dict[str, Any]:
    features = input_geojson.get("features", [])
    school_index = llm_result.get("school_feature_index")
    if isinstance(school_index, str) and school_index.isdigit():
        school_index = int(school_index)
    if not isinstance(school_index, int):
        school_index = None

    road_indices = [int(i) for i in llm_result.get("selected_road_feature_indices", []) if isinstance(i, int) or str(i).isdigit()]
    support_indices = [int(i) for i in llm_result.get("selected_supporting_feature_indices", []) if isinstance(i, int) or str(i).isdigit()]

    polygon = llm_result.get("school_region_polygon")
    if not polygon:
        polygon = fallback_polygon(input_geojson, road_indices, school_index)

    out_features: List[Dict[str, Any]] = []
    used: List[int] = []

    if school_index is not None and 0 <= school_index < len(features):
        school_feature = json.loads(json.dumps(features[school_index]))
        school_feature.setdefault("properties", {})
        school_feature["properties"].update({
            "output_role": "school_point",
            "source_feature_index": school_index,
        })
        out_features.append(school_feature)
        used.append(school_index)

    for idx in road_indices:
        if idx in used or idx < 0 or idx >= len(features):
            continue
        feature = features[idx]
        if feature.get("geometry", {}).get("type") != "LineString":
            continue
        road_feature = json.loads(json.dumps(feature))
        road_feature.setdefault("properties", {})
        road_feature["properties"].update({
            "output_role": "school_related_road",
            "source_feature_index": idx,
        })
        out_features.append(road_feature)
        used.append(idx)

    for idx in support_indices:
        if idx in used or idx < 0 or idx >= len(features):
            continue
        support_feature = json.loads(json.dumps(features[idx]))
        support_feature.setdefault("properties", {})
        support_feature["properties"].update({
            "output_role": "supporting_feature",
            "source_feature_index": idx,
        })
        out_features.append(support_feature)
        used.append(idx)

    if polygon is not None:
        out_features.append({
            "type": "Feature",
            "geometry": polygon,
            "properties": {
                "output_role": "inferred_school_region",
                "source_feature_index": None,
                "style_stroke_color": "#FF3B30",
                "style_stroke_width": 2,
                "style_fill_color": "#00000000",
                "style_fill_opacity": 0.0,
            },
        })

    return {
        "type": "FeatureCollection",
        "features": out_features,
        "metadata": {
            "generated_by": "Gemini via LangChain (local_guy_ai)",
            "model": GEMINI_MODEL,
            "input_path": str(INPUT_GEOJSON),
            "output_path": str(OUTPUT_GEOJSON),
            "confidence": llm_result.get("confidence"),
            "school_label": llm_result.get("school_label"),
            "reasoning": llm_result.get("reasoning"),
            "notes": llm_result.get("notes", []),
        },
    }


def run_local_gemini_call() -> Dict[str, Any]:
    api_key = get_gemini_api_key()
    input_geojson = read_geojson(INPUT_GEOJSON)
    prompt_text = build_prompt(input_geojson)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Return only strict JSON matching the requested schema."),
        ("user", "{prompt_text}"),
    ])
    llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        temperature=0.0,
        google_api_key=api_key,
    )

    chain = prompt | llm
    response = chain.invoke({"prompt_text": prompt_text})
    response_text = normalize_llm_text(getattr(response, "content", response))
    llm_result = extract_json_object(response_text)

    output_geojson = build_output_geojson(input_geojson, llm_result)

    with OUTPUT_RESPONSE.open("w", encoding="utf-8") as handle:
        json.dump(llm_result, handle, ensure_ascii=True, indent=2)
    with OUTPUT_GEOJSON.open("w", encoding="utf-8") as handle:
        json.dump(output_geojson, handle, ensure_ascii=True, indent=2)

    return {
        "input_path": str(INPUT_GEOJSON),
        "output_geojson": str(OUTPUT_GEOJSON),
        "output_response": str(OUTPUT_RESPONSE),
        "selected_road_count": len(llm_result.get("selected_road_feature_indices", [])),
        "selected_supporting_count": len(llm_result.get("selected_supporting_feature_indices", [])),
        "confidence": llm_result.get("confidence"),
        "school_label": llm_result.get("school_label"),
    }


print("Local Gemini helpers ready.")

Local Gemini helpers ready.


In [3]:
summary = run_local_gemini_call()
pd.DataFrame([summary])

,input_path,output_geojson,output_response,selected_road_count,selected_supporting_count,confidence,school_label
0,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,24,7,0.95,School Zone near Dessauerstraße and Gallwitzallee


In [4]:
geo = read_geojson(OUTPUT_GEOJSON)
rows = []
for feature in geo.get("features", []):
    rows.append({
        "output_role": feature.get("properties", {}).get("output_role"),
        "geometry_type": feature.get("geometry", {}).get("type"),
        "source_feature_index": feature.get("properties", {}).get("source_feature_index"),
        "fill_opacity": feature.get("properties", {}).get("style_fill_opacity"),
    })
pd.DataFrame(rows)

,output_role,geometry_type,source_feature_index,fill_opacity
0,school_point,Point,0.0,NaN
1,school_related_road,LineString,11.0,NaN
2,school_related_road,LineString,14.0,NaN
3,school_related_road,LineString,19.0,NaN
4,school_related_road,LineString,24.0,NaN
5,school_related_road,LineString,43.0,NaN
6,school_related_road,LineString,64.0,NaN
7,school_related_road,LineString,96.0,NaN
8,school_related_road,LineString,106.0,NaN
9,school_related_road,LineString,122.0,NaN
